# 🌐 MQ Topology Simplification & Analytical Dashboard
## Senior AI/ML Engineer & Data Architect View
--- 
**Objective:** Deep exploratory data analysis (EDA) and feature importance analysis to identify enterprise MQ connectivity inefficiencies and complexity.

### 📘 1. Setup & Data Loading
We use industry-standard libraries for data science: `pandas` for data manipulation, `networkx` for graph-based topology modeling, and `scikit-learn` for machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Set aesthetic parameters for visualizations
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams['figure.figsize'] = (12, 8)

def load_and_validate(file_path):
    """Dynamically load CSV and perform initial validation."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"CSV not found at {file_path}")
    
    df = pd.read_csv(file_path)
    # Handle missing values: replace empty strings with NaN and drop columns with 100% NaN
    df = df.replace(r'^\\s*$', np.nan, regex=True).dropna(axis=1, how='all')
    return df

# File path relative to the scratch directory
data_path = "MQ.Raw.Data.cleaned_v002.csv"
df_raw = load_and_validate(data_path)

print(f"Dataset Shape: {df_raw.shape}")
df_raw.head()

### 📊 2. Data Understanding (EDA)
Analyzing dataset schemas and volume-based insights.

In [ ]:
def basic_eda(df):
    """Generates summary of dataset statistics."""
    summary = pd.DataFrame({
        'Missing Values': df.isnull().sum(),
        'Unique Values': df.nunique(),
        'Data Type': df.dtypes
    })
    return summary

print("--- Data Summary ---")
print(basic_eda(df_raw))

# Domain-Specific Counts
print(f"\nUnique Queue Managers: {df_raw['queue_manager_name'].nunique()}")
print(f"Total Applications: {df_raw['app_id'].nunique()}")
print(f"Unique Queues: {df_raw['Discrete Queue Name'].nunique()}")

# Producers vs Consumers count
role_counts = df_raw.groupby('PrimaryAppRole').size()
print(f"\nApplication Roles Profile:\n{role_counts}")

### 🌐 3. Topology Modeling (Graph-Based)
Modeling the relationships between Queue Managers as a directed graph. We prioritize identifying hotspots and bottlenecks via centrality measures.

In [ ]:
# Pre-processing for Topology: Edges represent QM flows (Local -> Remote QM)
edges_df = df_raw[df_raw['remote_q_mgr_name'].notna() & (df_raw['remote_q_mgr_name'] != "")]

G = nx.DiGraph()
for idx, row in edges_df.iterrows():
    G.add_edge(row['queue_manager_name'], row['remote_q_mgr_name'], weight=1, label=row['app_id'])

# --- 3.1 Network Visualization ---
plt.title("MQ Interconnect Topology Map")
pos = nx.spring_layout(G, k=0.5)
nx.draw(G, pos, with_labels=True, node_color='skyblue', edge_color='gray', 
        node_size=2000, font_size=10, font_weight='bold', arrows=True)
plt.show()

# --- 3.2 Centrality Measures ---
degree_centrality = nx.degree_centrality(G)
betweenness = nx.betweenness_centrality(G)
closeness = nx.closeness_centrality(G)

topology_stats = pd.DataFrame({
    'Degree Centrality': degree_centrality,
    'Betweenness Centrality': betweenness,
    'Closeness Centrality': closeness
}).sort_values('Degree Centrality', ascending=False)

print("\n--- Higher Connectivity Nodes (Hotspots) ---")
print(topology_stats.head(5))

### 🔁 4. Custom MQ Complexity Metrics
We define a **Custom MQ Complexity Score** based on:
- Degree Centrality (Connectivity density)
- Queue per QM Ratio (Load density)
- Cycle detection (Circular routing risks)
- Redundant channel counts

In [ ]:
def compute_complexity_score(row, stats_df):
    """
    Custom score logic: 
    Score = Scaling(Degree) * 0.4 + Scaling(Load) * 0.3 + Scaling(Betweenness) * 0.3 
    """
    # Placeholder for logic mapping QM stats to the raw dataframe
    qm = row['queue_manager_name']
    if qm in stats_df.index:
        centrality = stats_df.loc[qm, 'Degree Centrality']
        between = stats_df.loc[qm, 'Betweenness Centrality']
        return (centrality * 10) + (between * 20)
    return 0

cycles = list(nx.simple_cycles(G))
print(f"\nDetected Circular Routing Cycles: {len(cycles)}")
if cycles: print(f"Sample Cycles: {cycles[:3]}")

### 🧠 5. Feature Engineering
Derived metrics that feed into our identification of overloaded QMs.

In [ ]:
# Aggregating Load per QM
qm_load = df_raw.groupby('queue_manager_name').agg({
    'Discrete Queue Name': 'count',
    'app_id': 'nunique',
    'line_of_business': 'nunique'
}).rename(columns={'Discrete Queue Name': 'queue_count', 'app_id': 'unique_apps', 'line_of_business': 'lob_count'})

# Merge graph features back with Load data
qm_features = qm_load.join(topology_stats).fillna(0)
qm_features['complexity_score'] = qm_features.apply(lambda r: (r['queue_count']*0.1) + (r['Degree Centrality']*5) + (r['Betweenness Centrality']*10), axis=1)

print("\n--- Engineered QM Meta-Features ---")
print(qm_features.head())

### 📈 6. Feature Importance Analysis
Predicting if a QM is a **'Complexity Hotspot'** and identifying which variables (Queues, Degree, etc.) drive this classification.

In [ ]:
# Define Target Proxy: High Complexity (Top 30 percentile)
threshold = qm_features['complexity_score'].quantile(0.7)
qm_features['Is_High_Complexity'] = (qm_features['complexity_score'] >= threshold).astype(int)

X = qm_features.drop(columns=['complexity_score', 'Is_High_Complexity'])
y = qm_features['Is_High_Complexity']

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

sns.barplot(x=importances.values, y=importances.index)
plt.title("Key Drivers of QM Topology Complexity")
plt.show()

### 🔍 7. Pattern & Anomaly Detection
Using unsupervised learning (K-Means) to cluster QMs into categories like 'Critical Hubs', 'Isolated Processing', and 'Redundant Forwarders'.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
qm_features['Cluster'] = kmeans.fit_predict(X_scaled)

plt.scatter(qm_features['queue_count'], qm_features['Degree Centrality'], c=qm_features['Cluster'], cmap='rainbow')
plt.xlabel("Count of Queues")
plt.ylabel("Degree Centrality")
plt.title("Cluster Analysis: QM Operating Archetypes")
plt.show()

### 📌 8. Key Insights & 📊 9. Dashboard Summary
- **Critical Nodes:** Nodes with high betweenness are architectural bottlenecks.
- **Concentrated Risk:** Identify QMs handling more than 5 LOBs or 100+ Applications.
- **Channel Entropy:** Excessive fan-in/fan-out ratios suggestive of point-to-point debt.
- **Simplification Slot:** QMs in the 'Isolated' cluster with low queue counts are candidates for consolidation.

In [ ]:
top_hubs = qm_features.nlargest(5, 'Betweenness Centrality').index.tolist()
consolidate_candidates = qm_features[(qm_features['queue_count'] < 5) & (qm_features['unique_apps'] < 2)].index.tolist()

print(f"Critical Consolidation Candidates: {consolidate_candidates}")
print(f"Key Interconnect Hubs (Critical Infrastructure): {top_hubs}")

# Export topology graph as JSON for high-fidelity visualization in Next.js/D3
import json
from networkx.readwrite import json_graph
data = json_graph.node_link_data(G)
with open('mq_topology_graph.json', 'w') as f:
    json.dump(data, f, indent=4)
print("\n--- Graph exported to mq_topology_graph.json ---")